# Copy Brain OS Demo: Scoring Persuasive Copy with TRIBE v2 Brain Predictions

**[Copy Brain OS](workflow/README.md)** extends [TRIBE v2](https://github.com/facebookresearch/tribev2) with a persuasion-focused brain scoring system.

TRIBE v2 predicts fMRI brain responses to naturalistic text, audio and video. Copy Brain OS maps those predictions onto **7 neuro-persuasion filters** derived from 15 peer-reviewed frameworks:

| Filter | Brain region (proxy) | Theoretical foundation |
|--------|---------------------|------------------------|
| **F1 Atenção** | Insula + ACC + Visual cortex | Loewenstein Gap Theory; Friston Prediction Error |
| **F2 Emoção** | Insula + OFC + Temporal pole | Damasio Somatic Marker; LeDoux Amygdala |
| **F3 Memória** | Parahippocampal + Fusiform | Paivio Dual Coding; Heath SUCCESs |
| **F4 Decisão** | Superior frontal + Precuneus | Kahneman Prospect Theory; Cialdini |
| **F5 Linguagem** | Broca + Wernicke + Angular | Flesch Readability; Sweller Cognitive Load |
| **F6 Recompensa** | OFC + Frontal pole + PCC | Schultz Dopamine; Ainslie Hyperbolic Discounting |
| **F7 Predição** | PCC + Precuneus + TPJ + ACC | Friston Free Energy; Clark Predictive Processing |

In this notebook we will:
1. Load TRIBE v2 + Destrieux atlas
2. Score a sample copy against all 7 filters
3. Compare two versions of a copy (A/B neural test)
4. Analyse per-section (HOOK → OFERTA/CTA) filter breakdown
5. Visualise scores as radar chart, time series, and brain surface

> **License note:** TRIBE v2 is released under CC BY-NC-4.0 — **non-commercial use only**. This notebook and all outputs share that restriction.

## Setup (for Colab users)

1. Activate the GPU (Menu > Runtime > Change runtime)
2. Run the cell below
3. Restart your environment

In [ ]:
# Colab / fresh install
# !uv pip install "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"
# !pip install nilearn plotly

## 1. Load TRIBE v2 + Copy Brain Scorer

We load the pretrained TRIBE v2 model from [HuggingFace Hub](https://huggingface.co/facebook/tribev2) and initialise the Copy Brain scorer.

On first run this downloads the model checkpoint (~1 GB). Subsequent runs use the cache.

> **Note:** you must accept the LLaMA 3.2-3B terms at [meta-llama/Llama-3.2-3B](https://huggingface.co/meta-llama/Llama-3.2-3B) and run `huggingface-cli login` once.

In [ ]:
from pathlib import Path
from tribev2.copy_brain import CopyBrainScorer
from tribev2.plotting import PlotBrain

CACHE_FOLDER = Path("./cache")

# Loads TRIBE v2 + Destrieux atlas parcellation
scorer = CopyBrainScorer.load(
    checkpoint_dir="facebook/tribev2",
    cache_folder=CACHE_FOLDER,
)

# Brain surface plotter (same as tribe_demo.ipynb)
plotter = PlotBrain(mesh="fsaverage5")

print("Scorer and plotter ready.")

## 2. Score a piece of copy

TRIBE v2 converts text → gTTS audio → word events → fMRI predictions.
Copy Brain maps those predictions to F1–F7 filter scores.

The example below uses a high-ticket B2B copy hook (implante odontológico).
Replace it with any copy you want to evaluate.

In [ ]:
copy_text = """
Se você está lendo isso, sua agenda já deveria estar cheia de pacientes de implante.
Mas ela não está. E você sabe o porquê.

Não é falta de competência. Não é falta de equipamento.
É que os pacientes que chegam até você não têm como pagar.
Você fecha consultas de R$200 enquanto concorrentes cobram R$8.000 pelo mesmo procedimento.

A diferença não está na técnica. Está em quem você está atraindo.

O Implant Patient Acquisition System™ mudou isso para 47 clínicas em 2024.
Em média, 12 pacientes qualificados financeiramente por mês, nos primeiros 30 dias.
Sem leads genéricos. Sem pacientes que somem no orçamento.
Só consultas com quem já decidiu investir no próprio sorriso.

Dr. Carlos, São Paulo: saiu de 2 para 11 implantes por mês em 6 semanas.
Dra. Ana, Belo Horizonte: faturou R$47.000 no primeiro mês do sistema.

Garantia: 8 pacientes qualificados em 30 dias ou devolvemos 100% do investimento.

Agenda com 3 vagas esta semana. Reserve a sua antes que feche.
Clique abaixo e veja como funciona na sua cidade.
"""

result = scorer.score_copy(copy_text, segment=True)
print(result.summary())

## 3. Visualise: Radar Chart

The radar chart shows the 7 filter scores (blue) against the minimum thresholds (orange dashed).
Any filter below threshold is a rewrite priority.

In [ ]:
import matplotlib.pyplot as plt
from tribev2.copy_brain import plot_filter_radar

fig, ax = plot_filter_radar(result)
plt.show()

## 4. Visualise: Filter Time Series

Each subplot shows how one filter's activation evolves over the duration of the copy.
TRIBE v2 predicts brain activity at each word group — green areas = above threshold, red = below.

In [ ]:
from tribev2.copy_brain import plot_filter_timeseries

fig, axes = plot_filter_timeseries(result, figsize=(14, 10))
plt.show()

## 5. Section Heatmap

The heatmap shows filter activation per funnel section.
Rows = F1–F7, Columns = HOOK / PROBLEMA / SOLUÇÃO / PROVA / OFERTA_CTA.

Dark red = weak. Dark green = strong. Diagonal pattern is ideal
(HOOK strong on F1/F7, OFERTA strong on F4/F6/F7).

In [ ]:
from tribev2.copy_brain import plot_section_heatmap

fig, ax = plot_section_heatmap(result)
plt.show()

## 6. A/B Neural Copy Test

Compare two versions of a hook. The scorer runs TRIBE v2 inference on both
and returns per-filter deltas (positive = Version B wins on that filter).

In [ ]:
hook_a = """
Descubra como clínicas odontológicas estão aumentando seus faturamentos.
Com nosso sistema, você pode conseguir mais pacientes de implante.
Entre em contato para saber mais sobre nossos serviços.
"""

hook_b = """
Se você está lendo isso, sua agenda já deveria estar cheia de pacientes de implante.
Mas ela não está. E você sabe o porquê.
Não é falta de competência — é quem você está atraindo.
"""

comparison = scorer.compare_versions(
    hook_a, hook_b,
    labels=("Hook A (genérico)", "Hook B (específico)")
)

print(f"Winner: {comparison['winner']}")
print(f"Mean activation: {comparison['mean_activation']}")
print(f"Improvement: +{comparison['improvement']:.3f}")
print("\nPer-filter deltas (+ = Hook B wins):")
for filt, delta in comparison['per_filter_delta'].items():
    sign = "+" if delta >= 0 else ""
    print(f"  {filt:<20s} {sign}{delta:.3f}")

## 7. Brain Surface: F1 Attention regions

Visualise which cortical vertices were activated by a specific filter
when the copy was processed. This uses the same `PlotBrain` API as `tribe_demo.ipynb`.

Below we show **F1 Atenção** (Insula + ACC + visual cortex).

In [ ]:
import numpy as np
from tribev2.copy_brain import plot_filter_on_brain

# We need the raw TRIBE v2 predictions — re-run inference to get them
# (In a real session, store preds from scorer._text_to_preds directly)
preds = scorer._text_to_preds(copy_text)
print(f"Predictions shape: {preds.shape}  (n_timesteps, n_vertices)")

fig = plot_filter_on_brain(
    filter_key="F1_Atencao",
    preds=preds,
    plotter=plotter,
    views=["left", "right", "dorsal"],
)
plt.show()

## 8. Export result as JSON

The result object serialises to JSON for downstream use in dashboards or automated pipelines.

In [ ]:
import json

result_dict = result.to_dict()
print(json.dumps(result_dict, indent=2, ensure_ascii=False)[:1500], "...")

## Next steps

- **Manual audit**: follow the full audit protocol in [`workflow/auditoria-completa.md`](workflow/auditoria-completa.md)
- **Quick checklists**: [`workflow/checklists/copy.md`](workflow/checklists/copy.md) (15 min) and [`workflow/checklists/roteiro.md`](workflow/checklists/roteiro.md) (30 min)
- **Filter deep dives**: [`workflow/filtros/`](workflow/filtros/) — one file per filter with criteria, examples, errors
- **References**: [`workflow/referencias/psicologia-base.md`](workflow/referencias/psicologia-base.md) — 15 frameworks with copy applications

---
*Brain Copy OS — built on TRIBE v2 (Meta FAIR, CC BY-NC-4.0). Non-commercial use only.*